# Pointwise LightGBM + SVD Standalone Evaluation

Binary classification model predicting `booking_bool`. A fundamentally different
approach from the listwise LambdaRank: treats each hotel-search pair
independently rather than optimizing ranking within each search group.

Also evaluates SVD latent factors as a standalone ranking signal (assignment
requirement for a Recommender Systems technique).

In [1]:
import sys
from pathlib import Path

cwd = Path(".").resolve()
REPO_ROOT = cwd if (cwd / ".git").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

import json
from itertools import product

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.utils import load_train, train_val_split, compute_ndcg_fast, make_submission
from src.features import build_features
from src.svd_features import build_svd_features

In [2]:
# === GPU AUTO-DETECTION ===
# Try GPU, fall back to CPU if unavailable.
def _detect_gpu():
    """Probe LightGBM GPU support with a tiny dataset."""
    try:
        import numpy as _np
        import lightgbm as _lgb
        _ds = _lgb.Dataset(_np.array([[0, 1], [1, 0]]), label=_np.array([0, 1]))
        _lgb.train({"objective": "binary", "device": "gpu", "verbose": -1, "num_threads": 1},
                   _ds, num_boost_round=1)
        return True
    except Exception:
        return False

GPU_AVAILABLE = _detect_gpu()
DEVICE_PARAMS = {"device": "gpu", "gpu_use_dp": False} if GPU_AVAILABLE else {}
if GPU_AVAILABLE:
    print("GPU mode enabled — using LightGBM GPU acceleration")
else:
    print("CPU mode — no compatible GPU detected")

GPU mode enabled — using LightGBM GPU acceleration


## Load Data + Feature Engineering

In [3]:
df = load_train()
n_raw = df.shape[1]
df = build_features(df)
print(f"Loaded {len(df):,} rows, {df['srch_id'].nunique():,} searches")
print(f"Columns: {n_raw} raw + {df.shape[1] - n_raw} base features = {df.shape[1]} total")

Loaded 4,958,347 rows, 199,795 searches
Columns: 54 raw + 40 base features = 94 total


## Train/Val Split + SVD Features

In [4]:
train_df, val_df = train_val_split(df)
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}")

# Build SVD features from training fold only (leakage-safe)
n_before = df.shape[1]
df = build_svd_features(train_df, df)
print(f"SVD features added: {df.shape[1] - n_before} columns")

# Re-split to get augmented train/val (same random_state=42 gives same split)
train_df, val_df = train_val_split(df)

Train: 3,966,682  Val: 991,665
SVD features added: 43 columns


## Prepare Features and Target

In [5]:
# Target: binary booking, NOT graded relevance
y_train = train_df["booking_bool"].values
y_val = val_df["booking_bool"].values

# Same exclusions as lambdarank notebook
DROP_COLS = [
    "srch_id", "date_time", "position", "click_bool", "booking_bool",
    "gross_bookings_usd", "random_bool",
]
COMP_RAW = [f"comp{i}_{s}" for i in range(1, 9) for s in ("rate", "inv", "rate_percent_diff")]
drop = set(DROP_COLS + COMP_RAW)
feature_cols = [c for c in train_df.columns if c not in drop]

print(f"Features: {len(feature_cols)}")

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

Features: 106


## Initial Training

In [6]:
params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 127,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "min_child_samples": 50,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

# No group= argument — pointwise treats each row independently
train_set = lgb.Dataset(X_train, label=y_train)
val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

model = lgb.train(
    params,
    train_set,
    valid_sets=[val_set],
    num_boost_round=4000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.110728
[200]	valid_0's binary_logloss: 0.109807
[300]	valid_0's binary_logloss: 0.109518
[400]	valid_0's binary_logloss: 0.109381
[500]	valid_0's binary_logloss: 0.109294
[600]	valid_0's binary_logloss: 0.109209
[700]	valid_0's binary_logloss: 0.109169
[800]	valid_0's binary_logloss: 0.109105
[900]	valid_0's binary_logloss: 0.109087
[1000]	valid_0's binary_logloss: 0.109059
[1100]	valid_0's binary_logloss: 0.109019
Early stopping, best iteration is:
[1149]	valid_0's binary_logloss: 0.108992


## Evaluation

In [7]:
val_df_eval = val_df.copy()
val_df_eval["pred_score"] = model.predict(X_val)

baseline_ndcg = compute_ndcg_fast(val_df_eval, "prop_location_score1")
pointwise_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")

print(f"Baseline NDCG@5 (location_score1): {baseline_ndcg:.4f}")
print(f"Pointwise LightGBM NDCG@5:         {pointwise_ndcg:.4f}")
print(f"LambdaRank NDCG@5:                 0.3820")
print(f"Improvement over baseline:          +{pointwise_ndcg - baseline_ndcg:.4f}")

Baseline NDCG@5 (location_score1): 0.1727
Pointwise LightGBM NDCG@5:         0.3893
LambdaRank NDCG@5:                 0.3820
Improvement over baseline:          +0.2167


## Feature Importance

In [8]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("Top 20 features by gain (pointwise):")
print(importance.head(20).to_string(index=False))

# Compare with LambdaRank if available
lambdarank_results = REPO_ROOT / "outputs" / "lambdarank_run1.json"
if lambdarank_results.exists():
    with open(lambdarank_results) as f:
        lr_data = json.load(f)
    if "top_features" in lr_data:
        print("\n\nLambdaRank top features (for comparison):")
        for feat, gain in list(lr_data["top_features"].items())[:10]:
            print(f"  {feat}: {gain:.1f}")

Top 20 features by gain (pointwise):
                         feature          gain
       prop_location_score2_rank 201529.612549
            prop_location_score2 188432.425873
              search_hotel_count 167716.472231
      hotel_book_rate_by_country 114359.536304
           price_usd_diff_median  91923.375747
                  price_usd_rank  90493.386506
            prop_location_score1  88959.695876
            prop_starrating_rank  88835.696158
       prop_starrating_diff_mean  77942.269638
             price_usd_diff_mean  68886.076676
             price_vs_historical  66589.306360
prop_location_score2_diff_median  57557.976571
                       price_usd  57212.709824
          prop_review_score_rank  55556.705139
        prop_location_score2_std  53146.899006
         hotel_book_rate_by_site  52763.101329
             srch_booking_window  50631.181873
       orig_destination_distance  49627.524893
        prop_location_score2_min  48215.893337
     prop_review_score_

## Hyperparameter Tuning

Same grid as LambdaRank. Evaluate by **NDCG@5** (competition metric), not logloss.

In [9]:
search_space = {
    "num_leaves": [63, 127, 255],
    "learning_rate": [0.01, 0.05, 0.1],
    "min_child_samples": [20, 50, 100],
}

results = []
base_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

keys = list(search_space.keys())
for combo in product(*search_space.values()):
    trial_params = {**base_params, **dict(zip(keys, combo))}

    train_set_hp = lgb.Dataset(X_train, label=y_train)
    val_set_hp = lgb.Dataset(X_val, label=y_val, reference=train_set_hp)

    m = lgb.train(
        trial_params,
        train_set_hp,
        valid_sets=[val_set_hp],
        num_boost_round=4000,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )

    preds = m.predict(X_val)
    val_df_eval["pred_score"] = preds
    score = compute_ndcg_fast(val_df_eval, "pred_score")

    row = dict(zip(keys, combo))
    row["ndcg5"] = score
    row["n_trees"] = m.best_iteration
    results.append(row)
    print(f"  leaves={row['num_leaves']:>3}  lr={row['learning_rate']:.2f}  "
          f"min_child={row['min_child_samples']:>3}  -> NDCG@5={score:.4f}  trees={m.best_iteration}")

results_df = pd.DataFrame(results).sort_values("ndcg5", ascending=False)
print("\nTop 5 configurations:")
print(results_df.head().to_string(index=False))

  leaves= 63  lr=0.01  min_child= 20  -> NDCG@5=0.3880  trees=4000
  leaves= 63  lr=0.01  min_child= 50  -> NDCG@5=0.3890  trees=4000
  leaves= 63  lr=0.01  min_child=100  -> NDCG@5=0.3899  trees=4000
  leaves= 63  lr=0.05  min_child= 20  -> NDCG@5=0.3872  trees=1758
  leaves= 63  lr=0.05  min_child= 50  -> NDCG@5=0.3895  trees=1790
  leaves= 63  lr=0.05  min_child=100  -> NDCG@5=0.3908  trees=2328
  leaves= 63  lr=0.10  min_child= 20  -> NDCG@5=0.3806  trees=366
  leaves= 63  lr=0.10  min_child= 50  -> NDCG@5=0.3831  trees=414
  leaves= 63  lr=0.10  min_child=100  -> NDCG@5=0.3856  trees=666
  leaves=127  lr=0.01  min_child= 20  -> NDCG@5=0.3916  trees=4000
  leaves=127  lr=0.01  min_child= 50  -> NDCG@5=0.3929  trees=4000
  leaves=127  lr=0.01  min_child=100  -> NDCG@5=0.3941  trees=4000
  leaves=127  lr=0.05  min_child= 20  -> NDCG@5=0.3874  trees=1138
  leaves=127  lr=0.05  min_child= 50  -> NDCG@5=0.3897  trees=1320
  leaves=127  lr=0.05  min_child=100  -> NDCG@5=0.3908  trees=137

## Retrain with Best Params

In [10]:
best = results_df.iloc[0]
print(f"Best config: leaves={int(best.num_leaves)}, lr={best.learning_rate}, "
      f"min_child={int(best.min_child_samples)}, NDCG@5={best.ndcg5:.4f}")

best_params = {
    **base_params,
    "num_leaves": int(best.num_leaves),
    "learning_rate": best.learning_rate,
    "min_child_samples": int(best.min_child_samples),
}

train_set_best = lgb.Dataset(X_train, label=y_train)
val_set_best = lgb.Dataset(X_val, label=y_val, reference=train_set_best)

best_model = lgb.train(
    best_params,
    train_set_best,
    valid_sets=[val_set_best],
    num_boost_round=4000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

val_df_eval["pred_score"] = best_model.predict(X_val)
final_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"\nFinal Pointwise NDCG@5: {final_ndcg:.4f}")

# Save model
models_dir = REPO_ROOT / "models"
models_dir.mkdir(exist_ok=True)
best_model.save_model(str(models_dir / "pointwise_best.txt"))

Best config: leaves=255, lr=0.01, min_child=100, NDCG@5=0.3954
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.115003
[200]	valid_0's binary_logloss: 0.112254
[300]	valid_0's binary_logloss: 0.111147
[400]	valid_0's binary_logloss: 0.110511
[500]	valid_0's binary_logloss: 0.110073
[600]	valid_0's binary_logloss: 0.109767
[700]	valid_0's binary_logloss: 0.109543
[800]	valid_0's binary_logloss: 0.109347
[900]	valid_0's binary_logloss: 0.109211
[1000]	valid_0's binary_logloss: 0.109095
[1100]	valid_0's binary_logloss: 0.109006
[1200]	valid_0's binary_logloss: 0.108919
[1300]	valid_0's binary_logloss: 0.108843
[1400]	valid_0's binary_logloss: 0.108782
[1500]	valid_0's binary_logloss: 0.108737
[1600]	valid_0's binary_logloss: 0.108692
[1700]	valid_0's binary_logloss: 0.108649
[1800]	valid_0's binary_logloss: 0.108613
[1900]	valid_0's binary_logloss: 0.108577
[2000]	valid_0's binary_logloss: 0.108544
[2100]	valid_0's binary_logloss: 0.108512
[22

## Save Results

In [11]:
out_dir = REPO_ROOT / "outputs"
out_dir.mkdir(exist_ok=True)

# Results JSON
results_out = {
    "model": "pointwise",
    "n_rows": len(df),
    "n_searches": int(df["srch_id"].nunique()),
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_features": len(feature_cols),
    "n_trees": int(best_model.best_iteration),
    "best_params": {k: v for k, v in best_params.items()
                    if k not in ("verbose", "num_threads", "seed")},
    "baseline_ndcg5": round(baseline_ndcg, 4),
    "model_ndcg5": round(final_ndcg, 4),
    "top_features": {
        row["feature"]: round(row["gain"], 1)
        for _, row in importance.head(15).iterrows()
    },
}

with open(out_dir / "pointwise_run1.json", "w") as f:
    json.dump(results_out, f, indent=2)
print("Results saved to outputs/pointwise_run1.json")

# Save val predictions for ensemble
val_preds = val_df_eval[["srch_id", "prop_id", "pred_score"]].copy()
val_preds.to_parquet(out_dir / "pointwise_val_preds.parquet", index=False)
print(f"Val predictions saved: {len(val_preds):,} rows")

Results saved to outputs/pointwise_run1.json
Val predictions saved: 991,665 rows


## SVD Standalone Evaluation

Evaluate SVD affinity score as a standalone ranking signal. This satisfies
the assignment requirement for a Recommender Systems technique.

In [12]:
ndcg_svd = compute_ndcg_fast(val_df_eval, "svd_affinity_score")

print("=" * 50)
print("Model Comparison (NDCG@5)")
print("=" * 50)
print(f"  Baseline (location_score1):  {baseline_ndcg:.4f}")
print(f"  SVD standalone:              {ndcg_svd:.4f}")
print(f"  Pointwise LightGBM:          {final_ndcg:.4f}")
print(f"  LambdaRank:                  0.3820")
print("=" * 50)

# Save SVD standalone results
svd_out = {
    "model": "svd_standalone",
    "n_components": 20,
    "ndcg5": round(ndcg_svd, 4),
    "baseline_ndcg5": round(baseline_ndcg, 4),
}
with open(out_dir / "svd_standalone.json", "w") as f:
    json.dump(svd_out, f, indent=2)
print(f"\nSVD standalone results saved to outputs/svd_standalone.json")

Model Comparison (NDCG@5)
  Baseline (location_score1):  0.1727
  SVD standalone:              0.2196
  Pointwise LightGBM:          0.3964
  LambdaRank:                  0.3820

SVD standalone results saved to outputs/svd_standalone.json


### Why SVD Underperforms as a Standalone Ranker

Collaborative filtering assumes a dense user-item interaction matrix with
persistent users. This dataset violates both assumptions:

- **Sparse, session-based interactions:** Each search is anonymous — there
  are no repeat-user IDs. The interaction matrix is very sparse.
- **Ignores search context:** SVD captures only destination-hotel affinity
  patterns. It completely ignores price, booking window, guest count,
  time of year, and all other search-level features.

The value of SVD in this project is primarily as a **feature source** (the 43
columns merged into LightGBM models) rather than a standalone ranker. The
`svd_affinity_score` adds signal that the tree models can combine with
price, location, and search context features.

This evaluation exists to satisfy the assignment requirement for "at least
one technique from the Recommender Systems lecture" and to demonstrate
understanding of when CF works and when it doesn't.